# Sesi 3 - RAG dengan OpenAI Embeddings
Notebook ini mengikuti project OpenAI Embeddings only. Tidak memakai TF-IDF.

## 1. Import library dan load environment

In [ ]:
from pathlib import Path
import os, sys

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

from rag_engine import load_all_documents, make_chunks, OpenAIEmbeddingVectorStore, build_context_block, answer_from_context, late_orders_summary

## 2. Load dokumen

In [ ]:
docs = load_all_documents(ROOT / "data" / "raw_docs")
print("Total dokumen:", len(docs))
print(docs[0]["metadata"])

## 3. Chunking dokumen

In [ ]:
chunks = make_chunks(docs, chunk_size=900, overlap=150)
print("Total chunk:", len(chunks))
print(chunks[0].metadata)
print(chunks[0].text[:300])

## 4. Build index OpenAI Embeddings
Cell ini membutuhkan `OPENAI_API_KEY`. Jalankan sekali saja, karena embedding disimpan ke JSON.

In [ ]:
index_path = ROOT / "data" / "vector_store" / "openai_embeddings_index.json"

if not os.getenv("OPENAI_API_KEY"):
    print("OPENAI_API_KEY belum ada. Isi file .env terlebih dahulu.")
else:
    store = OpenAIEmbeddingVectorStore(index_path=index_path)
    store.build(chunks)
    store.save()
    print("Index disimpan ke:", index_path)

## 5. Retrieval pertanyaan

In [ ]:
question = "Kalau order internal masuk saat weekend, kapan batas SLA-nya?"

if not index_path.exists():
    print("Index belum ada. Jalankan cell build index terlebih dahulu.")
elif not os.getenv("OPENAI_API_KEY"):
    print("OPENAI_API_KEY belum ada. Query embedding butuh API key.")
else:
    store = OpenAIEmbeddingVectorStore(index_path=index_path).load()
    contexts = store.search(question, k=4)
    print(build_context_block(contexts))
    print("\nJAWABAN:")
    print(answer_from_context(question, contexts))

## 6. SQL summary

In [ ]:
late_orders_summary(ROOT / "data" / "retail_bi.db")